# 01.1 — Why RAG exists

A language model knows what it read during training and nothing else. That's
obvious when you say it, and easy to forget when the model answers fluently
about something it has never seen.

This notebook makes it concrete, and then fixes it in one line.

In [1]:
!pip install -q openai==3.8.0

## Ask a question the model cannot possibly know

Sahel Microfinance Bank doesn't exist. It's one of the fictional organisations
in this course's corpus, invented for teaching. There is no chance the model
encountered it in training.

So we know for certain what it *should* say: I don't know.

In [4]:
import os
from openai import OpenAI

# In Docker this comes from .env. On Colab, use the secrets panel:
#     from google.colab import userdata
#     os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')

client = OpenAI(
    api_key=os.environ['OPENROUTER_API_KEY'],
    base_url='https://openrouter.ai/api/v1',
)

CHAT_MODEL = 'minimax/minimax-m2.7:free' # Check https://openrouter.ai/models for free models


def ask(prompt):
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
    )
    return response.choices[0].message.content

In [5]:
question = (
    'What is the maximum value of an emergency procurement that Sahel '
    'Microfinance Bank Plc can authorise without competitive sourcing?'
)

print(ask(question))

Based on the regulatory framework for Nigerian financial institutions, the maximum value for emergency procurement without competitive sourcing is generally guided by the Central Bank of Nigeria (CBN) Procurement Code and the bank's own internal procurement policy.

However, since I do not have access to the specific, current internal procurement policy of **Sahel Microfinance Bank Plc**, the most reliable reference is the **CBN Revised Procurement Code (2023)**, which sets the following benchmark for emergency procurements:

*   **Threshold:** Emergency procurement without competitive bidding may be undertaken for values **up to ₦5,000,000 (Five Million Naira)**.
*   **Condition:** This is strictly for cases of urgent and compelling need where competitive sourcing is impractical.

**Important Considerations:**
1.  **Internal Policy:** Sahel Microfinance Bank Plc's own procurement policy may set a **lower internal threshold** than the CBN's benchmark. Banks often adopt more stringent l

Run that a few times and see what you get.

Sometimes the model says it doesn't have information about this bank, which is
the correct answer. Sometimes it produces a figure, or explains what a typical
microfinance bank's threshold might be, or asks a clarifying question.

The useful thing to notice is not that it might get it wrong. It's that **you
cannot tell from the answer alone**. A confident, well-written paragraph looks
identical whether it came from knowledge or from pattern-matching on what a
procurement policy usually says.

The real answer is NGN 5,000,000. It's in section 4 of the bank's procurement
policy, which is sitting in `corpus/docs/` and which the model has never seen.

## Now give it the document

Same question. This time we paste the relevant paragraph into the prompt.

In [6]:
context = """
4. Emergency procurement

Where a failure of a critical service would result in branch closure, loss of
connectivity to the core banking application, or breach of a regulatory
deadline, the Head of Administration may authorise emergency procurement up to
NGN 5,000,000 without competitive sourcing. Emergency procurement must be
reported to the Management Procurement Committee at its next meeting with a
written justification.
"""

prompt = f"Answer using only the context below.\n\nCONTEXT:\n{context}\n\nQUESTION: {question}"

print(ask(prompt))

NGN 5,000,000


Correct, sourced, and reproducible.

**That is the whole idea.** Retrieval-augmented generation is: find the relevant
text, put it in the prompt, ask the question. Everything else in this course is
engineering around those three steps — because the hard part isn't the idea, it's
finding the right paragraph in ten thousand documents, quickly, without pasting
it by hand.

## Why not just train the model on the documents?

It's the obvious alternative and it's usually wrong. Four reasons.

**The knowledge is baked in.** When the procurement policy changes next quarter,
you retrain. With retrieval you replace a file.

**You can't see where an answer came from.** A fine-tuned model gives you a
sentence. Retrieval gives you a sentence and the document it came from, which is
the difference between an answer a compliance officer can use and one they can't.

**Everyone gets the same model.** Fine-tuning absorbs all your documents into one
set of weights, so there is no way to let an engineer query the engineering
handbook while blocking them from the HR files. Retrieval filters at query time.
That single point rules out fine-tuning for most enterprise work.

**It costs more and it's slower to iterate.** Training runs versus dropping a PDF
in a folder.

Fine-tuning is the right tool when you want to change *how* a model responds —
its tone, its format, its adherence to a schema. It's the wrong tool for teaching
it *what is true this week*.

## What's next

Before building anything: the next notebook asks whether you need RAG at all.

The corpus you'll spend this course on is about 8,000 tokens. Modern models
accept a million. There is a real question about whether the entire apparatus
you are about to build is necessary, and it deserves an honest answer before you
spend a week on it.